<a href="https://colab.research.google.com/github/humptybigdump/inprogress/blob/main/05_1_ISE2025_WordVectors_BagOfWords_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**To adapt this notebook to your own needs** and to be able to edit it, please make a copy of your own. This works via "*File*" -> "*Save a copy ..*."


---



Some of the techniques mentioned in Sect. 2.10 of the ISE 2025 lecture are already implemented in the [python NLTK library](https://www.nltk.org/) and machine learning libraries, as e.g., [sci-kit-learn](https://scikit-learn.org/stable/).

# Word Vectors, Bag of Words, and TF-IDF
To represent natural language text (documents) for NLP related tasks, we need a way to represent text in a **numerical representation**. This can be achieved by mapping tokens of a text into a **vector space**.

In [ ]:
#First we have to import nltk and download a few required packages
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('words')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


True

To demonstrate this process, we start with a simple example text corpus of 10 different sentences.

In [ ]:
#this is our document corpus
documents = ["Rain lashed against the hotel window,",
             "blurring the neon glow of the sign across the street.",
             "Inside the hotel room, Tabea munched on a bruised apple.",
             "Every channel was a rerun, dull and uninteresting",
             "the art on the walls dull and uninspiring.",
             "Tabea pulled out her phone,",
             "the rain still lashing against the window.",
             "A few taps later on the phone,",
             "Tabea was lost in a world more exciting",
             "choosing the next movie marathon."
]
#pre-processing, i.e. lower case and removing punctuation
processed_docs = [doc.lower().replace(".","").replace(",","") for doc in documents]
processed_docs

['rain lashed against the hotel window',
 'blurring the neon glow of the sign across the street',
 'inside the hotel room tabea munched on a bruised apple',
 'every channel was a rerun dull and uninteresting',
 'the art on the walls dull and uninspiring',
 'tabea pulled out her phone',
 'the rain still lashing against the window',
 'a few taps later on the phone',
 'tabea was lost in a world more exciting',
 'choosing the next movie marathon']

A **Bag-of-Words representation** of a text is a simplification in terms of a **multiset of words, disregarding its grammar and word order**.
Let's create bag-of-words representations of our 10 sentences. The table below contains a row for each sentence and a column for each word in our 49 word vocabulary. The value of a cell determins the frequency of the term represented in the column.

In [ ]:
#use pandas for pretty tables
import pandas as pd
from collections import Counter #this makes counting easier
tf = pd.DataFrame([Counter(doc.split()) for doc in processed_docs]).fillna(0)
tf

,rain,lashed,against,the,hotel,window,blurring,neon,glow,of,...,later,lost,in,world,more,exciting,choosing,next,movie,marathon
0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,3.0,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1.0,0.0,1.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0


Next step would be **1-hot encoding**. However, we do a little short-cut here and create already vectors with **term frequencies** of the 10 sentences as a matrix of 10 vectors of length 49.

In [ ]:
#use sci-kit learn library to create a vector from text
from sklearn.feature_extraction.text import CountVectorizer
vec = CountVectorizer(token_pattern=r"\w+")
vec.fit(documents)
feature_names = vec.get_feature_names_out()
tf_matrix = vec.transform(processed_docs)
pd.DataFrame(tf_matrix.todense(), columns=feature_names)


,a,across,against,and,apple,art,blurring,bruised,channel,choosing,...,street,tabea,taps,the,uninspiring,uninteresting,walls,was,window,world
0,0,0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
1,0,1,0,0,0,0,1,0,0,0,...,1,0,0,3,0,0,0,0,0,0
2,1,0,0,0,1,0,0,1,0,0,...,0,1,0,1,0,0,0,0,0,0
3,1,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,1,0,1,0,0
4,0,0,0,1,0,1,0,0,0,0,...,0,0,0,2,1,0,1,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
6,0,0,1,0,0,0,0,0,0,0,...,0,0,0,2,0,0,0,0,1,0
7,1,0,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
8,1,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,1
9,0,0,0,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0


We know that term frequency vectors would not be a good choice to represent documents properly, since it would overemphasize the most frequently used words which might not well represent the specific meaning of a document. Therefore, we are using term frequency together with **indirect document frequency**, i.e. **tf-idf**. We normalize term frequency by dividing it by the number of documents in which the term occurs within a given corpus. Thereby, frequentlly occuring words that occur only in fewer documents will get a higher weight, since they are "important" to characterize the meaning of a document.

In [ ]:
#luckily, sci-kit learn already provides a tfidf vectorizer ;-)
from sklearn.feature_extraction.text import TfidfVectorizer
# The options ensure that the numbers match our example above.
vec = TfidfVectorizer(smooth_idf=False, norm=None)
vec.fit(processed_docs)
feature_names = vec.get_feature_names_out()
tfidf_matrix = vec.transform(processed_docs)
pd.DataFrame(tfidf_matrix.todense(), columns=feature_names)

,across,against,and,apple,art,blurring,bruised,channel,choosing,dull,...,street,tabea,taps,the,uninspiring,uninteresting,walls,was,window,world
0,0.000000,2.609438,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,1.356675,0.000000,0.000000,0.000000,0.000000,2.609438,0.000000
1,3.302585,0.000000,0.000000,0.000000,0.000000,3.302585,0.000000,0.000000,0.000000,0.000000,...,3.302585,0.000000,0.000000,4.070025,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,3.302585,0.000000,0.000000,3.302585,0.000000,0.000000,0.000000,...,0.000000,2.203973,0.000000,1.356675,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,2.609438,0.000000,0.000000,0.000000,0.000000,3.302585,0.000000,2.609438,...,0.000000,0.000000,0.000000,0.000000,0.000000,3.302585,0.000000,2.609438,0.000000,0.000000
4,0.000000,0.000000,2.609438,0.000000,3.302585,0.000000,0.000000,0.000000,0.000000,2.609438,...,0.000000,0.000000,0.000000,2.713350,3.302585,0.000000,3.302585,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,2.203973,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.000000,2.609438,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,2.713350,0.000000,0.000000,0.000000,0.000000,2.609438,0.000000
7,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,3.302585,1.356675,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,2.203973,0.000000,0.000000,0.000000,0.000000,0.000000,2.609438,0.000000,3.302585
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.302585,0.000000,...,0.000000,0.000000,0.000000,1.356675,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


We can use tf-idf vectors of the documents to determine their "semantic" similarity by computing the pairwise **cosine similarity**. The matrix printed below shows the pairwise cosine similarity of the 10 sentences.

In [ ]:
#sci-kit learn also provides a cosine similarity method
from sklearn.metrics.pairwise import cosine_similarity
# compute and print the cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
pd.DataFrame(cosine_sim)

,0,1,2,3,4,5,6,7,8,9
0,1.000000,0.090591,0.160212,0.000000,0.076074,0.000000,0.541340,0.042811,0.000000,0.043167
1,0.090591,1.000000,0.067091,0.000000,0.149714,0.000000,0.162668,0.084251,0.000000,0.084952
2,0.160212,0.067091,1.000000,0.000000,0.130685,0.085392,0.061215,0.115380,0.069922,0.031969
3,0.000000,0.000000,0.000000,1.000000,0.222355,0.000000,0.000000,0.000000,0.104563,0.000000
4,0.076074,0.149714,0.130685,0.222355,1.000000,0.000000,0.136602,0.164110,0.000000,0.071339
5,0.000000,0.000000,0.085392,0.000000,0.000000,1.000000,0.000000,0.150317,0.089608,0.000000
6,0.541340,0.162668,0.061215,0.000000,0.136602,0.000000,1.000000,0.076872,0.000000,0.077512
7,0.042811,0.084251,0.115380,0.000000,0.164110,0.150317,0.076872,1.000000,0.000000,0.040146
8,0.000000,0.000000,0.069922,0.104563,0.000000,0.089608,0.000000,0.000000,1.000000,0.000000
9,0.043167,0.084952,0.031969,0.000000,0.071339,0.000000,0.077512,0.040146,0.000000,1.000000


We can observe, that there is a (relatively) high similarity between sentences 0 and 6:</br>0:"`Rain lashed against the hotel window`" and</br>6:"`the rain still lashing against the window`"